# Proyecto de regresión: predicción de ventas globales de videojuegos de PS4

## 3.1 Comprensión del negocio

### Introducción al problema

El dataset contiene videojuegos de PlayStation 4 con información de año, género, publicador y ventas en distintas regiones. Cada registro representa un juego y permite analizar su desempeño comercial.

El fenómeno representado es la distribución de ventas de juegos de PS4 a nivel internacional. Analizarlo es útil para identificar patrones de éxito según región, género, año o empresa publicadora.

El problema consiste en estimar las ventas globales de un videojuego a partir de sus características disponibles. Es un problema de regresión porque la salida es un valor numérico continuo.

### Objetivo de la predicción

La variable a predecir es `Global_Sales`, tomada de la columna original `Global`. Representa las ventas globales del videojuego en millones de copias vendidas.

Predecir esta variable es importante porque las ventas globales resumen el éxito comercial de un juego. La predicción puede apoyar decisiones de marketing, distribución e inversión.

### Aplicación del modelo

El modelo podría ser usado por analistas, publicadores o equipos de marketing para estimar el rendimiento de un juego de PS4. Sus resultados ayudarían a decidir campañas, inventario, regiones prioritarias o inversión.

Una predicción incorrecta podría causar sobreinversión o falta de apoyo a un juego con potencial. Además, existen riesgos porque el dataset no incluye factores como reseñas, competencia, popularidad en redes o campañas promocionales.


## 3.2 Comprensión de los datos

En esta etapa se revisa la estructura original del dataset, sus atributos, problemas de calidad y primeras relaciones con `Global_Sales`.

### Descripción general

- **Nombre del dataset:** Video Games Sales Dataset, archivo `PS4_GamesSales.csv`.
- **Fuente:** Kaggle, dataset publicado como `sidtwr/videogames-sales-dataset`.
- **URL:** https://www.kaggle.com/datasets/sidtwr/videogames-sales-dataset
- **Unidad de análisis:** cada fila representa un videojuego de PlayStation 4.
- **Variable dependiente:** `Global_Sales`, obtenida a partir de la columna original `Global`.
- **Unidad de la variable dependiente:** millones de copias vendidas a nivel global.

El archivo contiene videojuegos de PS4 con año, género, publicador y ventas regionales/globales en millones de copias.


In [ ]:
# Importación de librerías principales
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
sns.set_theme(style='whitegrid', palette='Set2')


### Carga del dataset

El dataset se carga desde el archivo local `PS4_GamesSales.csv`. Se usa `latin1` por la codificación del archivo y se leen `N/A` y cadenas vacías como nulos.


In [ ]:
DATASET_NAME = 'Video Games Sales Dataset - PS4_GamesSales.csv'
DATASET_SOURCE = 'Kaggle: sidtwr/videogames-sales-dataset'
DATASET_URL = 'https://www.kaggle.com/datasets/sidtwr/videogames-sales-dataset'
csv_path = Path('PS4_GamesSales.csv')

dataset_raw = pd.read_csv(csv_path, encoding='latin1', na_values=['N/A', ''], keep_default_na=True)
dataset = dataset_raw.rename(columns={'Global': 'Global_Sales'}).copy()

print(f'Archivo cargado: {csv_path}')
dataset.head()


In [ ]:
# Dimensiones originales solicitadas
dataset.shape


In [ ]:
target_original = 'Global'
target = 'Global_Sales'

general_description = pd.DataFrame({
    'Elemento': [
        'Nombre del dataset', 'Fuente', 'URL', 'Número original de registros',
        'Número original de columnas', 'Número de variables independientes',
        'Variable dependiente', 'Contenido general'
    ],
    'Valor': [
        DATASET_NAME,
        DATASET_SOURCE,
        DATASET_URL,
        dataset_raw.shape[0],
        dataset_raw.shape[1],
        dataset_raw.shape[1] - 1,
        f'{target} (columna original: {target_original})',
        'Videojuegos de PS4 con año, género, publicador y ventas regionales/globales en millones de copias.'
    ]
})

general_description


**Interpretación:** el dataset es tabular y la variable objetivo es `Global_Sales`. Las ventas regionales se revisarán con cuidado porque están muy relacionadas con el total global.


### Descripción de atributos

La tabla resume los atributos originales, su tipo, dato en Python y nulos. La columna original `Global` se usará como `Global_Sales`.


In [ ]:
attribute_metadata = {
    'Game': {
        'Descripción': 'Nombre del videojuego de PlayStation 4.',
        'Tipo de atributo': 'Nominal'
    },
    'Year': {
        'Descripción': 'Año de lanzamiento del videojuego.',
        'Tipo de atributo': 'Discreto'
    },
    'Genre': {
        'Descripción': 'Género principal del videojuego.',
        'Tipo de atributo': 'Nominal'
    },
    'Publisher': {
        'Descripción': 'Empresa publicadora o distribuidora del videojuego.',
        'Tipo de atributo': 'Nominal'
    },
    'North America': {
        'Descripción': 'Ventas en Norteamérica, expresadas en millones de copias.',
        'Tipo de atributo': 'Continuo'
    },
    'Europe': {
        'Descripción': 'Ventas en Europa, expresadas en millones de copias.',
        'Tipo de atributo': 'Continuo'
    },
    'Japan': {
        'Descripción': 'Ventas en Japón, expresadas en millones de copias.',
        'Tipo de atributo': 'Continuo'
    },
    'Rest of World': {
        'Descripción': 'Ventas en el resto del mundo, expresadas en millones de copias.',
        'Tipo de atributo': 'Continuo'
    },
    'Global': {
        'Descripción': 'Ventas globales totales del videojuego, expresadas en millones de copias.',
        'Tipo de atributo': 'Continuo'
    }
}

attribute_table = pd.DataFrame({
    'Atributo': dataset_raw.columns,
    'Descripción': [attribute_metadata[col]['Descripción'] for col in dataset_raw.columns],
    'Tipo de atributo': [attribute_metadata[col]['Tipo de atributo'] for col in dataset_raw.columns],
    'Tipo de dato en Python': dataset_raw.dtypes.astype(str).values,
    'Nulos': dataset_raw.isna().sum().values,
    'Porcentaje de nulos': (dataset_raw.isna().mean().values * 100).round(2)
})

attribute_table


**Interpretación:** `Game`, `Genre` y `Publisher` son nominales; `Year` es discreta; las ventas son continuas. Los nulos en `Year` y `Publisher` deberán tratarse antes del modelado.


### Calidad de los datos

Se revisan duplicados, nulos, inválidos, inconsistencias, atípicos, baja variabilidad, identificadores y posible fuga de datos.


In [ ]:
regional_sales_cols = ['North America', 'Europe', 'Japan', 'Rest of World']
sales_cols = regional_sales_cols + [target]
numeric_cols = dataset.select_dtypes(include=np.number).columns.tolist()
categorical_cols = dataset.select_dtypes(exclude=np.number).columns.tolist()

duplicate_count = int(dataset.duplicated().sum())
null_summary = pd.DataFrame({
    'Nulos': dataset.isna().sum(),
    'Porcentaje': (dataset.isna().mean() * 100).round(2)
}).sort_values('Nulos', ascending=False)

invalid_years = dataset.loc[
    dataset['Year'].notna() & (~dataset['Year'].between(2013, 2020)),
    ['Game', 'Year']
]
negative_sales = (dataset[sales_cols] < 0).sum()
all_zero_sales_count = int((dataset[sales_cols].sum(axis=1) == 0).sum())

constant_cols = [col for col in dataset.columns if dataset[col].nunique(dropna=False) == 1]
low_variability = pd.DataFrame({
    'Valores únicos': dataset.nunique(dropna=False),
    'Proporción del valor más frecuente': [dataset[col].value_counts(dropna=False, normalize=True).iloc[0] for col in dataset.columns]
}).sort_values('Proporción del valor más frecuente', ascending=False)

identifier_candidates = pd.DataFrame({
    'Valores únicos': dataset.nunique(dropna=False),
    'Proporción de unicidad': (dataset.nunique(dropna=False) / len(dataset)).round(4)
}).sort_values('Proporción de unicidad', ascending=False)

regional_sum_difference = (dataset[regional_sales_cols].sum(axis=1) - dataset[target]).abs()
leakage_summary = pd.DataFrame({
    'Variable': regional_sales_cols,
    'Motivo de posible fuga': [
        'Es parte directa del cálculo de Global_Sales.' for _ in regional_sales_cols
    ]
})

outlier_rows = []
for col in numeric_cols:
    q1 = dataset[col].quantile(0.25)
    q3 = dataset[col].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outlier_count = int(((dataset[col] < lower_limit) | (dataset[col] > upper_limit)).sum())
    outlier_rows.append([col, q1, q3, iqr, lower_limit, upper_limit, outlier_count])

outlier_summary = pd.DataFrame(
    outlier_rows,
    columns=['Variable', 'Q1', 'Q3', 'IQR', 'Límite inferior', 'Límite superior', 'Atípicos IQR']
)

print(f'Registros duplicados: {duplicate_count}')
print(f'Filas con ventas totalmente en cero: {all_zero_sales_count}')
print(f'Diferencia máxima entre suma regional y Global_Sales: {regional_sum_difference.max():.4f}')
print(f'Filas con diferencia mayor a 0.02 por redondeo: {(regional_sum_difference > 0.02).sum()}')

display(null_summary)
display(outlier_summary)
display(low_variability.head(10))
display(identifier_candidates.head(10))
display(leakage_summary)


In [ ]:
quality_report = pd.DataFrame({
    'Aspecto revisado': [
        'Registros duplicados',
        'Valores nulos',
        'Valores inválidos',
        'Inconsistencias',
        'Tipos de datos incorrectos',
        'Valores atípicos',
        'Columnas constantes',
        'Columnas con poca variabilidad',
        'Identificadores sin utilidad predictiva',
        'Posible fuga de datos'
    ],
    'Evidencia en el dataset': [
        f'Se detectaron {duplicate_count} registros duplicados exactos.',
        'Year y Publisher presentan valores faltantes o N/A; se cuantifican en la tabla de nulos.',
        f'Se revisaron años fuera del rango esperado 2013-2020 y ventas negativas. Ventas negativas por columna: {negative_sales.to_dict()}.',
        f'La suma de ventas regionales coincide casi exactamente con Global_Sales; la diferencia máxima es {regional_sum_difference.max():.4f}, atribuible a redondeo.',
        'Year puede leerse como float64 por contener nulos, aunque conceptualmente es una variable discreta de año.',
        'Las columnas de ventas muestran atípicos por la presencia de videojuegos superventas frente a muchos títulos de ventas bajas.',
        f'Columnas constantes detectadas: {constant_cols if constant_cols else "ninguna"}.',
        'Varias columnas de ventas tienen alta concentración en valores bajos o cero, especialmente en juegos sin ventas reportadas.',
        'Game tiene una proporción de unicidad muy alta, por lo que funciona como identificador del producto.',
        'North America, Europe, Japan y Rest of World forman parte directa de Global_Sales.'
    ],
    'Cómo podría afectar al modelado': [
        'Los duplicados inflarían el peso de ciertos juegos y podrían sesgar validación y entrenamiento.',
        'Los nulos impiden entrenar varios algoritmos si no se imputan y pueden introducir sesgo si se eliminan sin criterio.',
        'Valores fuera de rango o negativos generarían relaciones falsas y errores de interpretación.',
        'Inconsistencias entre ventas regionales y globales podrían indicar errores de captura o redondeo.',
        'Un tipo incorrecto puede impedir escalado, PCA, correlación o codificación adecuada.',
        'Los atípicos pueden dominar métricas como RMSE y hacer que el modelo priorice pocos casos extremos.',
        'Una columna constante no aporta información y puede generar problemas en selección o escalado.',
        'Poca variabilidad reduce la capacidad de una variable para explicar diferencias en la variable objetivo.',
        'Un identificador con muchos valores únicos puede causar sobreajuste si se codifica de forma directa.',
        'Usar ventas regionales para predecir ventas globales puede producir métricas artificialmente altas porque el objetivo se calcula con esas columnas.'
    ]
})

quality_report


**Interpretación:** se observan nulos, muchos valores bajos o cero y posibles atípicos. `Game` funciona como identificador y las ventas regionales pueden provocar fuga de datos al estar ligadas al total global.


### Análisis exploratorio de datos

Se calculan estadísticas y gráficas para observar distribuciones, atípicos y relaciones con `Global_Sales`.


In [ ]:
# Estadísticas descriptivas generales
dataset.describe(include='all').T


In [ ]:
# Estadísticas descriptivas numéricas con percentiles
dataset[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T


**Interpretación:** las ventas tienen muchos valores bajos y pocos valores muy altos, por lo que la distribución está sesgada a la derecha.


In [ ]:
# Distribución de la variable dependiente
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(dataset[target], bins=35, kde=True, ax=axes[0], color='#3B82F6')
axes[0].set_title('Distribución de Global_Sales')
axes[0].set_xlabel('Ventas globales (millones)')
axes[0].set_ylabel('Frecuencia')

sns.boxplot(x=dataset[target], ax=axes[1], color='#F59E0B')
axes[1].set_title('Boxplot de Global_Sales')
axes[1].set_xlabel('Ventas globales (millones)')

plt.tight_layout()
plt.show()


**Interpretación:** `Global_Sales` concentra muchos juegos con ventas bajas y pocos superventas. Estos atípicos pueden influir en el entrenamiento y en las métricas.


In [ ]:
# Distribución de variables categóricas y año
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

dataset['Year'].value_counts(dropna=False).sort_index().plot(kind='bar', ax=axes[0], color='#10B981')
axes[0].set_title('Cantidad de juegos por año')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Cantidad')

genre_order = dataset['Genre'].value_counts().index
sns.countplot(data=dataset, y='Genre', order=genre_order, ax=axes[1], color='#6366F1')
axes[1].set_title('Cantidad de juegos por género')
axes[1].set_xlabel('Cantidad')
axes[1].set_ylabel('Género')

top_publishers = dataset['Publisher'].value_counts().head(15).sort_values()
top_publishers.plot(kind='barh', ax=axes[2], color='#EF4444')
axes[2].set_title('Top 15 publicadores por cantidad de juegos')
axes[2].set_xlabel('Cantidad')
axes[2].set_ylabel('Publicador')

plt.tight_layout()
plt.show()


**Interpretación:** algunos años, géneros y publicadores tienen más registros que otros. Esto puede afectar el aprendizaje de categorías poco representadas.


In [ ]:
# Matriz de correlación de variables numéricas
corr_cols = ['Year'] + sales_cols
corr_matrix = dataset[corr_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de correlación de variables numéricas')
plt.show()

corr_matrix[target].sort_values(ascending=False)


**Interpretación:** las ventas regionales se relacionan fuertemente con `Global_Sales`. `Year` suele tener una relación más débil con las ventas totales.


In [ ]:
# Relaciones entre ventas regionales y variable dependiente
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, col in zip(axes, regional_sales_cols):
    sns.scatterplot(data=dataset, x=col, y=target, ax=ax, alpha=0.65)
    ax.set_title(f'{col} vs {target}')
    ax.set_xlabel(f'{col} (millones)')
    ax.set_ylabel('Ventas globales (millones)')

plt.tight_layout()
plt.show()


**Interpretación:** al aumentar las ventas regionales también aumenta `Global_Sales`. Esta relación puede mejorar el ajuste, pero debe vigilarse por posible fuga de datos.


In [ ]:
# Relación entre variables categóricas y Global_Sales
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

genre_order_by_median = dataset.groupby('Genre')[target].median().sort_values(ascending=False).index
sns.boxplot(data=dataset, x=target, y='Genre', order=genre_order_by_median, ax=axes[0], color='#60A5FA')
axes[0].set_title('Global_Sales por género')
axes[0].set_xlabel('Ventas globales (millones)')
axes[0].set_ylabel('Género')

top_publishers_by_sales = dataset.groupby('Publisher')[target].sum().sort_values(ascending=False).head(15).sort_values()
top_publishers_by_sales.plot(kind='barh', ax=axes[1], color='#F97316')
axes[1].set_title('Top 15 publicadores por ventas globales acumuladas')
axes[1].set_xlabel('Ventas globales acumuladas (millones)')
axes[1].set_ylabel('Publicador')

plt.tight_layout()
plt.show()


**Interpretación:** géneros y publicadores muestran diferencias en ventas. Estas variables requieren codificación antes de entrenar modelos.


## 3.3 Preparación de los datos

En esta etapa se define `X` y `y`, se eliminan columnas no adecuadas para predecir, y se prepara un flujo de imputación, codificación y escalado sin ajustar transformaciones con datos de prueba.


### Definición de X y y

- `y`: variable dependiente `Global_Sales`, ventas globales en millones de copias.
- `X`: variables independientes usadas antes de conocer el resultado global.

`Global_Sales` es numérica continua porque puede tomar valores decimales de ventas, por ejemplo 0.06, 1.50 o 19.39 millones. Por ello, el problema corresponde a regresión.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target = 'Global_Sales'

identifier_cols = ['Game']
leakage_cols = ['North America', 'Europe', 'Japan', 'Rest of World']
removed_columns = identifier_cols + leakage_cols

feature_cols = [col for col in dataset.columns if col not in removed_columns + [target]]

X = dataset[feature_cols].copy()
y = dataset[target].copy()

removed_columns_report = pd.DataFrame({
    'Columna eliminada': removed_columns,
    'Motivo': [
        'Identificador del videojuego; alta unicidad y bajo valor predictivo general.',
        'Venta regional usada para calcular Global_Sales; genera fuga de datos.',
        'Venta regional usada para calcular Global_Sales; genera fuga de datos.',
        'Venta regional usada para calcular Global_Sales; genera fuga de datos.',
        'Venta regional usada para calcular Global_Sales; genera fuga de datos.'
    ]
})

print(f'X: {X.shape}')
print(f'y: {y.shape}')
display(X.head())
display(y.describe())
display(removed_columns_report)


**Justificación:** `Game` se elimina por ser identificador. Las ventas regionales se eliminan porque permiten conocer directamente `Global_Sales`, ya que el total global se obtiene a partir de esas regiones.


### Tratamiento de valores faltantes

Se identifican los nulos solo en las variables independientes. La variable objetivo no se modifica.


In [ ]:
missing_X = pd.DataFrame({
    'Nulos': X.isna().sum(),
    'Porcentaje': (X.isna().mean() * 100).round(2)
}).sort_values('Nulos', ascending=False)

missing_X


In [ ]:
imputation_plan = pd.DataFrame({
    'Variable': ['Year', 'Genre', 'Publisher'],
    'Tipo': ['Discreta numérica', 'Nominal', 'Nominal'],
    'Técnica': ['Mediana', 'Constante: Unknown', 'Constante: Unknown'],
    'Justificación': [
        'La mediana es robusta y mantiene un año representativo sin verse afectada por extremos.',
        'No tiene nulos, pero el pipeline queda preparado si aparecen nuevos datos faltantes.',
        'Evita asignar los publicadores faltantes al publicador más frecuente.'
    ]
})

imputation_plan


**Efecto esperado:** después de la imputación no deben quedar nulos en `X`. La imputación se aplicará dentro del pipeline y se ajustará solo con el conjunto de entrenamiento.


### Conversión de variables categóricas

`Genre` y `Publisher` son variables nominales, por lo que se usa `OneHotEncoder`. No se usa codificación ordinal porque asignaría un orden artificial entre géneros o publicadores.


### Escalado

`Year` se escala con `StandardScaler`. Esto es útil para PCA y modelos sensibles a escala o distancia. Las variables one-hot no se escalan porque ya quedan en 0 y 1. Los árboles no requieren escalado, pero usar el mismo preprocesamiento facilita comparar modelos.


In [ ]:
numeric_features = ['Year']
categorical_features = ['Genre', 'Publisher']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_transformer, numeric_features),
    ('categorical', categorical_transformer, categorical_features)
], remainder='drop', verbose_feature_names_out=False)

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

prepared_feature_names = preprocessor.get_feature_names_out()

X_train_prepared = pd.DataFrame(X_train_prepared, columns=prepared_feature_names, index=X_train.index)
X_test_prepared = pd.DataFrame(X_test_prepared, columns=prepared_feature_names, index=X_test.index)

print(f'Columnas antes de preparar X: {X.shape[1]}')
print(f'Columnas después de preparar X_train: {X_train_prepared.shape[1]}')
print(f'Nulos en X_train preparado: {int(X_train_prepared.isna().sum().sum())}')
print(f'Nulos en X_test preparado: {int(X_test_prepared.isna().sum().sum())}')

X_train_prepared.head()


In [ ]:
encoder = preprocessor.named_transformers_['categorical'].named_steps['encoder']

encoding_report = pd.DataFrame({
    'Variable categórica': categorical_features,
    'Tipo': ['Nominal', 'Nominal'],
    'Técnica': ['OneHotEncoder', 'OneHotEncoder'],
    'Columnas generadas': [len(categories) for categories in encoder.categories_],
    'Justificación': [
        'Representa géneros sin imponer jerarquía.',
        'Representa publicadores sin imponer jerarquía.'
    ]
})

display(encoding_report)
display(pd.DataFrame({
    'Etapa': ['Antes de codificar', 'Después de imputar, codificar y escalar'],
    'Número de columnas': [X.shape[1], X_train_prepared.shape[1]]
}))


### Resultado de la preparación

La preparación queda definida con un `ColumnTransformer`: imputa `Year` con mediana, imputa categóricas faltantes como `Unknown`, codifica `Genre` y `Publisher` con one-hot y escala `Year`. El ajuste se realiza con `X_train` y solo se transforma `X_test`, evitando fuga de información.


## 3.4 Selección y transformación de características

Se construyen dos representaciones de los datos: una con características seleccionadas y otra con componentes principales. Ambas parten de los datos ya preparados para evitar nulos, variables categóricas sin codificar y escalas incompatibles.


### Escenario 1: características seleccionadas

La selección final se hará con tres criterios: correlación con `y`, RFE e importancia mediante árboles. La correlación no se usa como único criterio porque solo mide relaciones lineales y no captura interacciones o relaciones no lineales.


#### Matriz de correlación

Se calcula la correlación entre cada característica preparada y `Global_Sales`, usando solo el conjunto de entrenamiento.


In [ ]:
corr_with_target = X_train_prepared.apply(lambda col: col.corr(y_train)).fillna(0)

correlation_table = pd.DataFrame({
    'Característica': corr_with_target.index,
    'Correlación con y': corr_with_target.values,
    'Correlación absoluta': corr_with_target.abs().values
}).sort_values('Correlación absoluta', ascending=False)

display(correlation_table.head(15))
display(correlation_table.tail(10))


In [ ]:
feature_corr_matrix = X_train_prepared.corr().abs()
upper_triangle = feature_corr_matrix.where(np.triu(np.ones(feature_corr_matrix.shape), k=1).astype(bool))

high_feature_corr = (
    upper_triangle.stack()
    .reset_index()
    .rename(columns={'level_0': 'Característica 1', 'level_1': 'Característica 2', 0: 'Correlación absoluta'})
    .sort_values('Correlación absoluta', ascending=False)
)

high_feature_corr[high_feature_corr['Correlación absoluta'] >= 0.80].head(10)


**Interpretación:** las correlaciones positivas indican características asociadas con mayores ventas; las negativas, con menores ventas. Algunas variables pueden tener poca relación lineal, pero aun así ser útiles en modelos no lineales. Las variables one-hot de una misma columna pueden presentar dependencia entre sí, por lo que también se revisa posible multicolinealidad.


#### Recursive Feature Elimination, RFE

Se aplica RFE con `LinearRegression`. Este estimador es compatible porque genera coeficientes (`coef_`) que RFE usa para eliminar características de menor contribución.


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

n_selected_features = 6
rfe_estimator = LinearRegression()
rfe = RFE(estimator=rfe_estimator, n_features_to_select=n_selected_features)
rfe.fit(X_train_prepared, y_train)

rfe_table = pd.DataFrame({
    'Característica': X_train_prepared.columns,
    'Ranking RFE': rfe.ranking_,
    'Seleccionada por RFE': rfe.support_
}).sort_values(['Ranking RFE', 'Característica'])

display(rfe_table.head(20))

rfe_selected_features = rfe_table.loc[rfe_table['Seleccionada por RFE'], 'Característica'].tolist()
rfe_discarded_features = rfe_table.loc[~rfe_table['Seleccionada por RFE'], 'Característica'].tolist()

print('Características seleccionadas por RFE:')
print(rfe_selected_features)
print('\nNúmero de características descartadas por RFE:', len(rfe_discarded_features))


#### Importancia mediante árboles

Se utiliza `RandomForestRegressor` porque captura relaciones no lineales y entrega importancias de características mediante `feature_importances_`.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

tree_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2
)
tree_model.fit(X_train_prepared, y_train)

tree_importance_table = pd.DataFrame({
    'Característica': X_train_prepared.columns,
    'Importancia en árboles': tree_model.feature_importances_
}).sort_values('Importancia en árboles', ascending=False)

display(tree_importance_table.head(15))


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=tree_importance_table.head(15),
    x='Importancia en árboles',
    y='Característica',
    color='#22C55E'
)
plt.title('Top 15 características por importancia en Random Forest')
plt.xlabel('Importancia')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()


**Interpretación:** las importancias de árboles permiten detectar características útiles aunque su relación con `Global_Sales` no sea estrictamente lineal. Estos resultados se comparan con correlación y RFE para elegir una selección final equilibrada.


#### Selección final

Se integran los tres métodos. Una característica se favorece si aparece bien posicionada en al menos dos criterios, o si tiene una importancia alta en árboles y aporta información interpretable.


In [ ]:
selection_comparison = (
    correlation_table[['Característica', 'Correlación con y', 'Correlación absoluta']]
    .merge(rfe_table[['Característica', 'Ranking RFE', 'Seleccionada por RFE']], on='Característica')
    .merge(tree_importance_table, on='Característica')
)

selection_comparison['Top correlación'] = selection_comparison['Correlación absoluta'].rank(ascending=False, method='min') <= 15
selection_comparison['Top árboles'] = selection_comparison['Importancia en árboles'].rank(ascending=False, method='min') <= 15
selection_comparison['Puntaje conjunto'] = (
    selection_comparison['Top correlación'].astype(int)
    + selection_comparison['Seleccionada por RFE'].astype(int)
    + selection_comparison['Top árboles'].astype(int)
)

selection_comparison = selection_comparison.sort_values(
    ['Puntaje conjunto', 'Importancia en árboles', 'Correlación absoluta'],
    ascending=[False, False, False]
)

selected_features = selection_comparison.head(n_selected_features)['Característica'].tolist()

selection_comparison['Decisión'] = np.where(
    selection_comparison['Característica'].isin(selected_features),
    'Incluir',
    'Excluir'
)
selection_comparison['Resultado'] = np.where(
    selection_comparison['Característica'].isin(selected_features),
    'Seleccionada',
    'No seleccionada'
)

final_selection_table = selection_comparison[[
    'Característica', 'Correlación con y', 'Ranking RFE', 'Seleccionada por RFE',
    'Importancia en árboles', 'Resultado', 'Decisión'
]]

display(final_selection_table.head(20))
print('Selección final:')
print(selected_features)


**Criterio final:** se incluyen entre cuatro y seis características con mejor evidencia conjunta. Se excluyen las demás por baja correlación, bajo ranking RFE, baja importancia en árboles o información redundante. Si los métodos difieren, se prioriza la coincidencia entre métodos y la importancia en árboles para capturar relaciones no lineales.


In [ ]:
X_train_selected = X_train_prepared[selected_features].copy()
X_test_selected = X_test_prepared[selected_features].copy()

print(f'X_train_selected: {X_train_selected.shape}')
print(f'X_test_selected: {X_test_selected.shape}')
X_train_selected.head()


### Escenario 2: Análisis de Componentes Principales, PCA

PCA se aplica después de imputar, codificar y escalar. Para evitar fuga de información, se incorpora en un pipeline y se ajusta únicamente con `X_train`.


In [ ]:
from sklearn.decomposition import PCA

pca_analysis_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(random_state=42))
])

pca_analysis_pipeline.fit(X_train, y_train)
pca_model = pca_analysis_pipeline.named_steps['pca']

explained_variance = pca_model.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
n_components_95 = int(np.argmax(cumulative_variance >= 0.95) + 1)

pca_variance_table = pd.DataFrame({
    'Componente': np.arange(1, len(explained_variance) + 1),
    'Varianza explicada': explained_variance,
    'Varianza acumulada': cumulative_variance
})

print(f'Número de variables antes de PCA: {X_train_prepared.shape[1]}')
print(f'Número de componentes generados: {len(explained_variance)}')
print(f'Componentes conservados para explicar al menos 95%: {n_components_95}')
display(pca_variance_table.head(20))


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(pca_variance_table['Componente'], pca_variance_table['Varianza acumulada'], marker='o', linewidth=2)
plt.axhline(0.95, color='red', linestyle='--', label='95% de varianza')
plt.axvline(n_components_95, color='green', linestyle='--', label=f'{n_components_95} componentes')
plt.title('Varianza explicada acumulada por PCA')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza explicada acumulada')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
pca_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=n_components_95, random_state=42))
])

X_train_pca = pca_pipeline.fit_transform(X_train, y_train)
X_test_pca = pca_pipeline.transform(X_test)

pca_columns = [f'PC{i}' for i in range(1, n_components_95 + 1)]
X_train_pca = pd.DataFrame(X_train_pca, columns=pca_columns, index=X_train.index)
X_test_pca = pd.DataFrame(X_test_pca, columns=pca_columns, index=X_test.index)

dimensionality_reduction = pd.DataFrame({
    'Representación': ['Antes de PCA', 'Después de PCA'],
    'Número de columnas': [X_train_prepared.shape[1], X_train_pca.shape[1]]
})

display(dimensionality_reduction)
X_train_pca.head()


**Justificación de PCA:** se conservan los componentes necesarios para explicar al menos 95% de la varianza. PCA reduce dimensionalidad y puede ayudar a modelos sensibles a escala o multicolinealidad. Su desventaja es que los componentes son combinaciones de variables, por lo que la interpretación del modelo se vuelve menos directa y puede perderse información de baja varianza.


## 3.5 Modelado

Se seleccionan tres algoritmos de regresión y se prueban en los dos escenarios definidos: características seleccionadas y PCA. Uno de los modelos es una red neuronal, como solicita el proyecto.


### Algoritmos seleccionados

Los tres algoritmos son diferentes entre sí: un modelo lineal, un modelo basado en árboles y una red neuronal.


In [ ]:
algorithm_description = pd.DataFrame({
    'Algoritmo': ['Regresión Lineal', 'Random Forest Regressor', 'MLPRegressor'],
    'Funcionamiento general': [
        'Ajusta una relación lineal entre las características y la variable objetivo.',
        'Construye varios árboles de decisión y promedia sus predicciones.',
        'Red neuronal multicapa que aprende relaciones no lineales mediante neuronas y pesos.'
    ],
    'Parámetros principales': [
        'fit_intercept=True',
        'n_estimators=150, min_samples_leaf=2, random_state=42',
        'hidden_layer_sizes=(32,16), activation=relu, max_iter=600, early_stopping=True'
    ],
    'Justificación': [
        'Sirve como modelo base simple e interpretable.',
        'Captura relaciones no lineales y es robusto ante variables one-hot.',
        'Cumple el requisito de red neuronal y puede aprender patrones no lineales.'
    ],
    'Ventajas': [
        'Rápida, interpretable y fácil de comparar.',
        'Buen desempeño en problemas tabulares y poca necesidad de supuestos lineales.',
        'Flexible para relaciones complejas entre variables.'
    ],
    'Limitaciones': [
        'No captura bien relaciones no lineales.',
        'Menos interpretable y puede tardar más que un modelo lineal.',
        'Sensible a escala, hiperparámetros y tamaño del dataset.'
    ]
})

algorithm_description


### Pipelines utilizados

Los seis experimentos usan la misma división de entrenamiento, la misma semilla y validación cruzada con 7 particiones. Los datos de prueba se conservan sin usarse para decidir parámetros.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import KFold, cross_validate
from sklearn.neural_network import MLPRegressor

RANDOM_STATE = 42

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, feature_names):
        self.feature_names = feature_names

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.reindex(columns=self.feature_names, fill_value=0)


model_definitions = {
    'Regresión Lineal': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    'Red Neuronal MLP': MLPRegressor(
        hidden_layer_sizes=(32, 16),
        activation='relu',
        solver='adam',
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=600,
        early_stopping=True,
        random_state=RANDOM_STATE
    )
}

def build_selected_features_pipeline(model):
    selected_preprocessor = clone(preprocessor).set_output(transform='pandas')
    return Pipeline(steps=[
        ('preprocessor', selected_preprocessor),
        ('feature_selector', FeatureSelector(selected_features)),
        ('model', model)
    ])

def build_pca_model_pipeline(model):
    return Pipeline(steps=[
        ('preprocessor', clone(preprocessor)),
        ('pca', PCA(n_components=n_components_95, random_state=RANDOM_STATE)),
        ('model', model)
    ])

pipeline_description = pd.DataFrame({
    'Escenario': ['Características seleccionadas', 'PCA'],
    'Pipeline': [
        'preprocessor -> feature_selector -> modelo',
        'preprocessor -> PCA -> modelo'
    ],
    'Descripción': [
        'Usa las 6 características elegidas por correlación, RFE e importancia de árboles.',
        f'Usa {n_components_95} componentes principales ajustados dentro del pipeline.'
    ]
})

pipeline_description


### Hiperparámetros

Los parámetros se fijan antes de evaluar con validación cruzada. No se usan los datos de prueba para elegir configuración.


In [ ]:
hyperparameter_report = pd.DataFrame({
    'Algoritmo': ['Regresión Lineal', 'Random Forest', 'Red Neuronal MLP'],
    'Parámetros modificados': [
        'No se modifican parámetros relevantes.',
        'n_estimators=150, min_samples_leaf=2, random_state=42',
        'hidden_layer_sizes=(32,16), alpha=0.001, max_iter=600, early_stopping=True'
    ],
    'Razón': [
        'Se usa como línea base.',
        'Aumentar estabilidad y reducir sobreajuste en hojas muy pequeñas.',
        'Controlar complejidad, permitir convergencia y detener entrenamiento si no mejora.'
    ],
    'Cómo se evitó fuga': [
        'Configuración definida antes de usar el conjunto de prueba.',
        'Configuración definida antes de usar el conjunto de prueba.',
        'Configuración definida antes de usar el conjunto de prueba.'
    ]
})

hyperparameter_report


### Validación cruzada de los seis experimentos

Se usa `KFold` con 7 particiones, `shuffle=True` y `random_state=42`. Las métricas son MAE, RMSE y R². También se registra el tiempo promedio de entrenamiento por partición.


In [ ]:
cv_strategy = KFold(n_splits=7, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

experiment_builders = {
    'Características seleccionadas': build_selected_features_pipeline,
    'PCA': build_pca_model_pipeline
}

experiment_results = []
experiment_pipelines = {}

for scenario_name, pipeline_builder in experiment_builders.items():
    for model_name, model in model_definitions.items():
        experiment_name = f'{scenario_name} - {model_name}'
        experiment_pipeline = pipeline_builder(clone(model))
        experiment_pipelines[experiment_name] = experiment_pipeline

        cv_scores = cross_validate(
            experiment_pipeline,
            X_train,
            y_train,
            cv=cv_strategy,
            scoring=scoring,
            n_jobs=1,
            return_train_score=False
        )

        experiment_results.append({
            'Escenario': scenario_name,
            'Modelo': model_name,
            'Experimento': experiment_name,
            'MAE promedio': -cv_scores['test_MAE'].mean(),
            'RMSE promedio': -cv_scores['test_RMSE'].mean(),
            'R2 promedio': cv_scores['test_R2'].mean(),
            'Tiempo entrenamiento promedio (s)': cv_scores['fit_time'].mean()
        })

modeling_results = pd.DataFrame(experiment_results).sort_values('RMSE promedio')
modeling_results


**Condiciones equivalentes:** los seis experimentos usan `X_train`, `y_train`, las mismas 7 particiones, la misma semilla y las mismas métricas. El conjunto de prueba queda reservado para la evaluación final del mejor modelo.


## 3.6 Evaluación y comparación de los seis experimentos

Los seis experimentos se evalúan con `KFold(n_splits=7, shuffle=True, random_state=42)`. Se usan las mismas particiones, la misma semilla y las mismas métricas: R² y MAE.


### Validación cruzada

Se guardan los resultados de cada partición para calcular media, desviación estándar y varianza.


In [ ]:
cv_detailed_rows = []
model_order = list(model_definitions.keys())

experiment_labels = {}
for scenario_name in experiment_builders.keys():
    for model_position, model_name in enumerate(model_order, start=1):
        if scenario_name == 'Características seleccionadas':
            label = f'Modelo {model_position} con características seleccionadas'
        else:
            label = f'Modelo {model_position} con PCA'
        experiment_labels[(scenario_name, model_name)] = label

for scenario_name, pipeline_builder in experiment_builders.items():
    for model_name, model in model_definitions.items():
        experiment_pipeline = pipeline_builder(clone(model))

        cv_scores = cross_validate(
            experiment_pipeline,
            X_train,
            y_train,
            cv=cv_strategy,
            scoring={'R2': 'r2', 'MAE': 'neg_mean_absolute_error'},
            n_jobs=1,
            return_train_score=False
        )

        for fold_index, (r2_value, mae_value, fit_time) in enumerate(
            zip(cv_scores['test_R2'], -cv_scores['test_MAE'], cv_scores['fit_time']),
            start=1
        ):
            cv_detailed_rows.append({
                'Escenario': scenario_name,
                'Modelo': model_name,
                'Experimento': experiment_labels[(scenario_name, model_name)],
                'Partición': fold_index,
                'R²': r2_value,
                'MAE': mae_value,
                'Tiempo entrenamiento (s)': fit_time
            })

cv_fold_results = pd.DataFrame(cv_detailed_rows)
cv_fold_results.head(10)


In [ ]:
cv_summary = (
    cv_fold_results
    .groupby(['Escenario', 'Modelo'], as_index=False)
    .agg(
        **{
            'R² medio': ('R²', 'mean'),
            'Desv. est. R²': ('R²', 'std'),
            'Varianza R²': ('R²', 'var'),
            'MAE medio': ('MAE', 'mean'),
            'Desv. est. MAE': ('MAE', 'std'),
            'Varianza MAE': ('MAE', 'var'),
            'Tiempo entrenamiento medio (s)': ('Tiempo entrenamiento (s)', 'mean')
        }
    )
)

cv_summary = cv_summary.sort_values(['R² medio', 'MAE medio'], ascending=[False, True])

selected_features_results = cv_summary[cv_summary['Escenario'] == 'Características seleccionadas'].copy()
pca_results = cv_summary[cv_summary['Escenario'] == 'PCA'].copy()
general_results = cv_summary[[
    'Escenario', 'Modelo', 'R² medio', 'Desv. est. R²', 'Varianza R²',
    'MAE medio', 'Desv. est. MAE', 'Varianza MAE'
]].copy()

display(selected_features_results)
display(pca_results)
display(general_results)


### Boxplot de comparación

El boxplot compara los valores de R² obtenidos en las siete particiones de cada experimento.


In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=cv_fold_results, x='Experimento', y='R²', color='#93C5FD')
sns.stripplot(data=cv_fold_results, x='Experimento', y='R²', color='#1F2937', size=4, alpha=0.7)
plt.axhline(0.80, color='red', linestyle='--', label='R² esperado = 0.80')
plt.title('Comparación de R² en validación cruzada')
plt.xlabel('Experimento')
plt.ylabel('R² por partición')
plt.xticks(rotation=35, ha='right')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
boxplot_analysis = (
    cv_fold_results
    .groupby(['Escenario', 'Modelo', 'Experimento'])
    .agg(
        Mediana_R2=('R²', 'median'),
        Q1_R2=('R²', lambda x: x.quantile(0.25)),
        Q3_R2=('R²', lambda x: x.quantile(0.75)),
        Min_R2=('R²', 'min'),
        Max_R2=('R²', 'max'),
        Desv_est_R2=('R²', 'std')
    )
    .reset_index()
)
boxplot_analysis['Rango intercuartílico'] = boxplot_analysis['Q3_R2'] - boxplot_analysis['Q1_R2']
boxplot_analysis['Rango total'] = boxplot_analysis['Max_R2'] - boxplot_analysis['Min_R2']
boxplot_analysis['Estabilidad'] = np.where(
    boxplot_analysis['Desv_est_R2'] <= 0.10,
    'Más estable',
    'Menos estable'
)

boxplot_analysis.sort_values('Mediana_R2', ascending=False)


**Interpretación del boxplot:** se comparan mediana, dispersión, rango intercuartílico y posibles particiones atípicas. Un modelo estable debe mantener R² similares entre particiones, sin depender de una sola partición con resultado alto.


### Desempeño esperado

Se espera un R² medio igual o superior a 0.80, desviación estándar reducida y varianza de R² preferentemente no superior a 0.01.


In [ ]:
best_by_metrics = cv_summary.sort_values(
    ['R² medio', 'MAE medio', 'Desv. est. R²', 'Tiempo entrenamiento medio (s)'],
    ascending=[False, True, True, True]
).iloc[0]

performance_check = pd.DataFrame({
    'Criterio': [
        'R² medio >= 0.80',
        'Varianza R² <= 0.01',
        'Desv. est. R² reducida',
        'MAE consistente',
        'Sin fuga de datos'
    ],
    'Resultado': [
        best_by_metrics['R² medio'] >= 0.80,
        best_by_metrics['Varianza R²'] <= 0.01,
        best_by_metrics['Desv. est. R²'] <= 0.10,
        best_by_metrics['Desv. est. MAE'] <= best_by_metrics['MAE medio'],
        True
    ],
    'Valor observado': [
        best_by_metrics['R² medio'],
        best_by_metrics['Varianza R²'],
        best_by_metrics['Desv. est. R²'],
        best_by_metrics['Desv. est. MAE'],
        'Se excluyeron Game y ventas regionales de X.'
    ]
})

performance_check


In [ ]:
if best_by_metrics['R² medio'] < 0.80:
    improvement_attempts = pd.DataFrame({
        'Revisión requerida': [
            'Preparación de datos',
            'Características utilizadas',
            'Hiperparámetros',
            'Efecto de PCA',
            'Limitación del dataset'
        ],
        'Acción documentada': [
            'Se imputaron nulos, se codificaron categóricas y se escaló Year dentro del pipeline.',
            'Se aplicaron correlación, RFE e importancia de árboles; se excluyeron variables con fuga.',
            'Se ajustaron parámetros de Random Forest y MLP antes de evaluar con prueba.',
            'Se comparó PCA contra características seleccionadas usando las mismas particiones.',
            'Al eliminar ventas regionales, el dataset queda con poca información predictiva directa.'
        ]
    })
else:
    improvement_attempts = pd.DataFrame({
        'Conclusión': ['El mejor experimento alcanza el umbral esperado de R² medio >= 0.80.']
    })

improvement_attempts


### Selección del mejor modelo

La selección considera R² medio, MAE medio, desviación estándar, varianza, estabilidad visual del boxplot, tiempo de entrenamiento, complejidad, interpretación y viabilidad de despliegue.


In [ ]:
best_selected = selected_features_results.sort_values(
    ['R² medio', 'MAE medio', 'Desv. est. R²'],
    ascending=[False, True, True]
).iloc[0]

best_pca = pca_results.sort_values(
    ['R² medio', 'MAE medio', 'Desv. est. R²'],
    ascending=[False, True, True]
).iloc[0]

best_overall = best_by_metrics

pca_effect = 'mejoró' if best_pca['R² medio'] > best_selected['R² medio'] else 'empeoró o no mejoró'

best_model_decision = pd.DataFrame({
    'Pregunta': [
        'Mejor modelo con características seleccionadas',
        'Mejor modelo con PCA',
        'Efecto de PCA',
        'Mejor combinación general',
        'Razón principal de selección'
    ],
    'Respuesta': [
        best_selected['Modelo'],
        best_pca['Modelo'],
        pca_effect,
        f"{best_overall['Escenario']} - {best_overall['Modelo']}",
        'Mayor R² medio con MAE competitivo y estabilidad entre particiones.'
    ]
})

best_model_decision


### Tabla detallada del mejor modelo

Se muestran los resultados del mejor experimento en las siete particiones y sus estadísticos principales.


In [ ]:
best_experiment_folds = cv_fold_results[
    (cv_fold_results['Escenario'] == best_overall['Escenario'])
    & (cv_fold_results['Modelo'] == best_overall['Modelo'])
].copy()

best_model_fold_table = best_experiment_folds[['Partición', 'R²', 'MAE']].copy()

summary_rows = pd.DataFrame({
    'Partición': ['Media', 'Desviación estándar', 'Varianza'],
    'R²': [
        best_experiment_folds['R²'].mean(),
        best_experiment_folds['R²'].std(),
        best_experiment_folds['R²'].var()
    ],
    'MAE': [
        best_experiment_folds['MAE'].mean(),
        best_experiment_folds['MAE'].std(),
        best_experiment_folds['MAE'].var()
    ]
})

best_model_fold_table = pd.concat([best_model_fold_table, summary_rows], ignore_index=True)
best_model_fold_table


In [ ]:
best_partition = best_experiment_folds.loc[best_experiment_folds['R²'].idxmax()]
worst_partition = best_experiment_folds.loc[best_experiment_folds['R²'].idxmin()]
r2_difference = best_partition['R²'] - worst_partition['R²']

q1 = best_experiment_folds['R²'].quantile(0.25)
q3 = best_experiment_folds['R²'].quantile(0.75)
iqr = q3 - q1
lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr
outlier_folds = best_experiment_folds[
    (best_experiment_folds['R²'] < lower_limit) | (best_experiment_folds['R²'] > upper_limit)
]

best_model_analysis = pd.DataFrame({
    'Aspecto': [
        'Mejor partición',
        'Peor partición',
        'Diferencia entre mejor y peor R²',
        'Estabilidad',
        'Particiones atípicas',
        'Representatividad de la media',
        'Magnitud de la varianza',
        'Confiabilidad general'
    ],
    'Resultado': [
        f"Partición {int(best_partition['Partición'])} con R²={best_partition['R²']:.4f}",
        f"Partición {int(worst_partition['Partición'])} con R²={worst_partition['R²']:.4f}",
        f'{r2_difference:.4f}',
        'Alta' if best_experiment_folds['R²'].std() <= 0.10 else 'Moderada o baja',
        'No se detectan' if outlier_folds.empty else ', '.join(outlier_folds['Partición'].astype(str)),
        'Adecuada' if best_experiment_folds['R²'].std() <= 0.10 else 'Revisar por variabilidad',
        f"{best_experiment_folds['R²'].var():.4f}",
        'Confiable si mantiene buen R², bajo MAE y baja variabilidad.'
    ]
})

best_model_analysis


## 3.7 Evaluación final del mejor modelo

Después de comparar los seis experimentos, se evalúa únicamente el mejor modelo con el conjunto de prueba reservado. Se mantiene la misma configuración de preprocesamiento, representación y algoritmo usada en validación cruzada.


### División de entrenamiento y prueba

La división ya se realizó con `train_test_split`: 80% entrenamiento, 20% prueba y `random_state=42`. El conjunto de prueba no se usó para selección de variables, PCA, ajuste de hiperparámetros ni validación cruzada.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_scenario = best_overall['Escenario']
best_model_name = best_overall['Modelo']

best_final_pipeline = experiment_builders[best_scenario](clone(model_definitions[best_model_name]))
best_final_pipeline.fit(X_train, y_train)

y_pred = best_final_pipeline.predict(X_test)
residuals = y_test - y_pred

final_test_metrics = pd.DataFrame({
    'Métrica': ['R²', 'MAE', 'RMSE'],
    'Resultado en prueba': [
        r2_score(y_test, y_pred),
        mean_absolute_error(y_test, y_pred),
        np.sqrt(mean_squared_error(y_test, y_pred))
    ]
})

print(f'Mejor experimento seleccionado: {best_scenario} - {best_model_name}')
print(f'Tamaño entrenamiento: {X_train.shape[0]} registros ({X_train.shape[0] / len(X):.0%})')
print(f'Tamaño prueba: {X_test.shape[0]} registros ({X_test.shape[0] / len(X):.0%})')
final_test_metrics


### Gráfica de valores reales contra valores predichos

La línea diagonal representa una predicción ideal: mientras más cerca estén los puntos de esa línea, menor es el error.


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.75, color='#2563EB')

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], color='red', linestyle='--', label='Predicción ideal')

plt.title('Valores reales vs valores predichos')
plt.xlabel('Global_Sales real')
plt.ylabel('Global_Sales predicho')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
prediction_analysis = pd.DataFrame({
    'Real': y_test,
    'Predicho': y_pred,
    'Residuo': residuals,
    'Error absoluto': np.abs(residuals)
})

prediction_analysis['Tipo de error'] = np.where(
    prediction_analysis['Residuo'] > 0,
    'Subestimación',
    np.where(prediction_analysis['Residuo'] < 0, 'Sobreestimación', 'Exacto')
)

prediction_analysis['Rango de venta real'] = pd.qcut(
    prediction_analysis['Real'],
    q=4,
    duplicates='drop'
)

error_by_sales_range = (
    prediction_analysis
    .groupby('Rango de venta real', observed=False)
    .agg(
        Registros=('Real', 'count'),
        Real_promedio=('Real', 'mean'),
        Predicho_promedio=('Predicho', 'mean'),
        MAE=('Error absoluto', 'mean'),
        Residuo_promedio=('Residuo', 'mean')
    )
    .reset_index()
)

display(prediction_analysis.sort_values('Error absoluto', ascending=False).head(10))
display(prediction_analysis['Tipo de error'].value_counts().rename_axis('Tipo de error').reset_index(name='Cantidad'))
display(error_by_sales_range)


**Análisis:** si los puntos se alejan de la línea ideal, el modelo comete más error. Los puntos por debajo de la línea representan subestimaciones y los puntos por encima sobreestimaciones. El resumen por rangos permite identificar si los errores crecen en juegos con ventas bajas, medias o altas.


### Gráfica de residuos

Los residuos se calculan como `valor real - valor predicho`. La línea horizontal en cero indica ausencia de error.


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.75, color='#7C3AED')
plt.axhline(0, color='red', linestyle='--', label='Residuo = 0')
plt.title('Residuos vs valores predichos')
plt.xlabel('Valores predichos')
plt.ylabel('Residuo (real - predicho)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
residual_summary = pd.DataFrame({
    'Estadístico': [
        'Media de residuos',
        'Mediana de residuos',
        'Desv. est. de residuos',
        'Residuo mínimo',
        'Residuo máximo',
        'Error absoluto medio',
        'Cantidad de errores extremos'
    ],
    'Valor': [
        residuals.mean(),
        residuals.median(),
        residuals.std(),
        residuals.min(),
        residuals.max(),
        np.abs(residuals).mean(),
        int((np.abs(residuals) > (np.abs(residuals).mean() + 2 * np.abs(residuals).std())).sum())
    ]
})

residual_by_pred_range = (
    prediction_analysis
    .assign(Rango_predicho=pd.qcut(prediction_analysis['Predicho'], q=4, duplicates='drop'))
    .groupby('Rango_predicho', observed=False)
    .agg(
        Registros=('Predicho', 'count'),
        Residuo_promedio=('Residuo', 'mean'),
        Dispersion_residuos=('Residuo', 'std'),
        MAE=('Error absoluto', 'mean')
    )
    .reset_index()
)

display(residual_summary)
display(residual_by_pred_range)


**Análisis de residuos:** una distribución adecuada debería estar centrada cerca de cero y sin patrones claros. Si los residuos aumentan o disminuyen conforme crece la predicción, hay indicios de sesgo o heterocedasticidad. Errores extremos pueden indicar juegos difíciles de predecir por información faltante en el dataset, como reseñas, popularidad, presupuesto o campañas de marketing.


### Conclusión de evaluación final

La evaluación final permite observar el comportamiento del mejor modelo en datos no usados para entrenar ni seleccionar el experimento. Las gráficas y tablas muestran si el modelo generaliza, si presenta sesgo sistemático y en qué rangos de ventas tiene mayor dificultad.


## 3.8 Despliegue del modelo

El mejor modelo se despliega con Flask y Render. La aplicación carga un pipeline completo guardado con `joblib`, muestra un formulario con variables originales y devuelve la predicción de `Global_Sales` en millones de copias.


### Entrenamiento y almacenamiento

Después de finalizar la evaluación, el pipeline del mejor experimento se vuelve a entrenar con todos los registros disponibles. El pipeline incluye imputación, codificación, escalado, PCA o selección de características y el algoritmo de regresión.


In [ ]:
import json
import shutil
import joblib

deployment_dir = Path('render_app')
deployment_dir.mkdir(exist_ok=True)

final_deployment_pipeline = experiment_builders[best_scenario](clone(model_definitions[best_model_name]))
final_deployment_pipeline.fit(X, y)

model_path = deployment_dir / 'model_pipeline.joblib'
metadata_path = deployment_dir / 'model_metadata.json'

joblib.dump(final_deployment_pipeline, model_path)

deployment_metadata = {
    'dataset': 'PS4_GamesSales.csv',
    'target': target,
    'target_unit': 'millones de copias vendidas',
    'input_features': feature_cols,
    'removed_columns': removed_columns,
    'best_scenario': str(best_scenario),
    'best_model': str(best_model_name),
    'selected_features': selected_features,
    'pca_components': int(n_components_95),
    'genres': sorted(dataset['Genre'].dropna().astype(str).unique().tolist()),
    'publishers': sorted(dataset['Publisher'].fillna('Unknown').astype(str).unique().tolist()),
}

if 'Unknown' not in deployment_metadata['publishers']:
    deployment_metadata['publishers'].insert(0, 'Unknown')

with open(metadata_path, 'w', encoding='utf-8') as file:
    json.dump(deployment_metadata, file, ensure_ascii=False, indent=2)

shutil.copyfile('PS4_GamesSales.csv', deployment_dir / 'PS4_GamesSales.csv')

url_file = Path('URL_Render.txt')
if not url_file.exists():
    url_file.write_text('Pendiente: pega aqui la URL publica de Render cuando el servicio este desplegado.', encoding='utf-8')

pd.DataFrame({
    'Archivo generado': [str(model_path), str(metadata_path), str(deployment_dir / 'PS4_GamesSales.csv'), str(url_file)],
    'Propósito': [
        'Pipeline final entrenado.',
        'Datos auxiliares para formulario y descripción del modelo.',
        'Dataset usado para reproducir entrenamiento.',
        'Archivo donde se colocará la URL pública de Render.'
    ]
})


### Servicio web con Flask

La carpeta `render_app` contiene la aplicación. El usuario ingresa `Year`, `Genre` y `Publisher`, que son variables originales del dataset. El usuario no ingresa valores escalados, variables one-hot ni componentes principales; el pipeline realiza esas transformaciones internamente.


In [ ]:
deployment_files = pd.DataFrame({
    'Archivo': [
        'render_app/app.py',
        'render_app/model_utils.py',
        'render_app/train_model.py',
        'render_app/model_pipeline.joblib',
        'render_app/model_metadata.json',
        'render_app/requirements.txt',
        'render_app/render.yaml',
        'render_app/.python-version',
        'render_app/templates/index.html',
        'render_app/static/styles.css'
    ],
    'Uso': [
        'Aplicación Flask: carga el pipeline, valida formulario y predice.',
        'Clase auxiliar usada por el pipeline guardado.',
        'Script reproducible para entrenar y guardar el modelo.',
        'Pipeline final entrenado.',
        'Categorías, escenario y modelo seleccionado.',
        'Dependencias necesarias en Render.',
        'Configuración opcional del servicio en Render.',
        'Versión de Python usada para el despliegue.',
        'Formulario HTML.',
        'Estilos de la interfaz.'
    ]
})

deployment_files


### Configuración en Render

En Render se debe crear un Web Service conectado al repositorio que contenga la carpeta `render_app`.

- **Build Command:** `pip install -r requirements.txt`
- **Start Command:** `gunicorn app:app`
- **Root Directory:** `render_app`, si el repositorio contiene también otros archivos.
- **URL:** se debe copiar en `URL_Render.txt` cuando Render la genere.


### Pruebas realizadas

La aplicación se probó localmente abriendo la página, enviando el formulario con datos reales del dataset y verificando que se generara una predicción sin errores. En Render deberán repetirse las mismas pruebas y capturar evidencias de la página, formulario y resultado.
